<a href="https://colab.research.google.com/github/daviddosu/daviddosu/blob/main/mutual_info_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, json, csv, time, random
from datetime import datetime
import requests
import pandas as pd

# ---------- Paths (customize if you like) ----------
STATE_FILE          = "qc_resume_state.json"    # JSON with {year, cursor, total_collected, ...}
STREAM_JSONL        = "qc_stream.jsonl"         # append-only raw works (one JSON object per line)
PROCESSED_IDS_TXT   = "qc_processed_ids.txt"    # one OpenAlex work id per line to avoid duplicate writes
NODES_CSV           = "qc_nodes.csv"            # node attributes (one row per work)
EDGES_CSV           = "qc_edges.csv"            # edges (citing_id → cited_id)
FINAL_JSON          = "qc_full.json"            # optional rolled-up JSON (from JSONL)
FINAL_FLAT_CSV      = "qc_flat_preview.csv"     # quick flattened preview (optional)

OPENALEX_URL = "https://api.openalex.org/works"

# — include all fields you’ll actually analyze downstream
SELECT_FIELDS = (
    "id,doi,title,publication_year,publication_date,"
    "cited_by_count,counts_by_year,"
    "concepts,authorships,primary_location,open_access,"
    "referenced_works,type,language"
)

# ---------- Utilities ----------
def print_ckpt(msg: str) -> None:
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] {msg}", flush=True)

def save_state(state: dict) -> None:
    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)
    print_ckpt(f"checkpoint saved → {STATE_FILE}")

def load_state(start_year: int, end_year: int) -> dict:
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, "r") as f:
            s = json.load(f)
        print_ckpt(f"resuming at year {s['year']}, cursor={repr(s['cursor'])}, total={s['total_collected']:,}")
        return s
    return {
        "year": start_year, "cursor": "*",
        "total_collected": 0,
        "start_year": start_year, "end_year": end_year,
        "stream_path": STREAM_JSONL
    }

def ensure_file_with_header(path: str, header: list) -> csv.writer:
    exists = os.path.exists(path)
    f = open(path, "a", newline="", encoding="utf-8")
    writer = csv.writer(f)
    if not exists or os.path.getsize(path) == 0:
        writer.writerow(header)
        f.flush()
    return f, writer

def load_processed_ids(path: str) -> set:
    s = set()
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                s.add(line.strip())
    print_ckpt(f"loaded {len(s):,} processed ids")
    return s

def append_processed_id(path: str, wid: str) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(wid + "\n")

def append_jsonl(path: str, objs: list) -> None:
    with open(path, "a", encoding="utf-8") as f:
        for o in objs:
            f.write(json.dumps(o, ensure_ascii=False) + "\n")

def polite_sleep(resp: requests.Response, base=0.2, max_cap=90):
    ra = resp.headers.get("Retry-After")
    if ra:
        try:
            wait = min(max_cap, int(ra))
            print_ckpt(f"Retry-After={wait}s — sleeping")
            time.sleep(wait + random.uniform(0, 1.5))
            return
        except ValueError:
            pass
    rem = resp.headers.get("x-ratelimit-remaining")
    reset = resp.headers.get("x-ratelimit-reset")
    try:
        rem_i = int(rem) if rem is not None else None
        reset_i = int(reset) if reset is not None else None
        if rem_i is not None and reset_i is not None and rem_i <= 1:
            print_ckpt(f"near limit: remaining={rem_i}, reset={reset_i}s — sleeping")
            time.sleep(min(max_cap, reset_i) + random.uniform(0, 1.5))
            return
    except ValueError:
        pass
    time.sleep(base + random.uniform(0, 0.3))

# ---------- Core fetcher ----------
def get_quantum_papers_resumable(
    concept_id="C58053490",
    start_year=2000,
    end_year=2024,
    email="david.dosu@cern.ch",
    per_page=200,
    checkpoint_each_page=True,
):
    assert per_page <= 200, "OpenAlex max per_page is 200."

    # session with retries
    session = requests.Session()
    try:
        from urllib3.util.retry import Retry
        from requests.adapters import HTTPAdapter
        retry = Retry(
            total=5, backoff_factor=0.5,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=frozenset(["GET"])
        )
        adapter = HTTPAdapter(max_retries=retry)
        session.mount("https://", adapter)
        session.mount("http://", adapter)
    except Exception:
        pass

    headers = {
        "User-Agent": f"OpenAlex-Client (mailto:{email})",
        "Accept": "application/json",
    }

    state = load_state(start_year, end_year)
    total_collected = state["total_collected"]
    stream_path = state.get("stream_path", STREAM_JSONL)
    if not os.path.exists(stream_path):
        open(stream_path, "w").close()

    # outputs — open once
    nodes_fh, nodes_writer = ensure_file_with_header(
        NODES_CSV,
        [
            "openalex_id","doi","title","pub_year","pub_date",
            "cited_by_count","counts_by_year_json",
            "venue_id","venue_name","venue_type",
            "type","language","is_oa",
            "institutions_json","concepts_json"
        ]
    )
    edges_fh, edges_writer = ensure_file_with_header(
        EDGES_CSV,
        ["citing_id","cited_id","citing_pub_year","citing_pub_date"]
    )

    processed_ids = load_processed_ids(PROCESSED_IDS_TXT)

    year   = state["year"]
    cursor = state["cursor"]
    grand_total_before = total_collected

    print_ckpt(f"collecting concept:{concept_id} {start_year}–{end_year} with cursor pagination")
    try:
        while year <= end_year:
            params = {
                "filter": f"concepts.id:{concept_id},publication_year:{year}",
                "per_page": per_page,
                "cursor": cursor,
                "select": SELECT_FIELDS,
                "mailto": email,
                "sort": "publication_date:asc"
            }
            r = session.get(OPENALEX_URL, params=params, headers=headers, timeout=60)

            if r.status_code == 429:
                print_ckpt(f"429 on year {year} cursor {repr(cursor)}")
                polite_sleep(r, base=3.0, max_cap=180)
                continue
            if r.status_code >= 500:
                print_ckpt(f"server {r.status_code}; backing off")
                polite_sleep(r, base=2.0, max_cap=120)
                continue

            r.raise_for_status()
            payload = r.json()
            results = payload.get("results", [])
            meta = payload.get("meta", {})
            next_cursor = meta.get("next_cursor")
            count = meta.get("count")

            # write raw
            if results:
                append_jsonl(stream_path, results)

            # process → nodes + edges
            wrote_nodes = wrote_edges = 0
            for w in results:
                wid = w.get("id")
                if not wid:
                    continue
                if wid in processed_ids:
                    continue

                pub_year = w.get("publication_year")
                pub_date = w.get("publication_date")
                oa       = (w.get("open_access") or {}).get("is_oa", False)
                venue    = ((w.get("primary_location") or {}).get("source")) or {}
                counts   = w.get("counts_by_year", [])
                # institutions — unique by id
                insts = []
                seen_inst = set()
                for a in w.get("authorships", []) or []:
                    for inst in a.get("institutions", []) or []:
                        iid = inst.get("id")
                        if iid and iid not in seen_inst:
                            seen_inst.add(iid)
                            insts.append({
                                "id": iid,
                                "display_name": inst.get("display_name"),
                                "country_code": inst.get("country_code"),
                                "type": inst.get("type"),
                            })

                nodes_writer.writerow([
                    wid, w.get("doi"), w.get("title"),
                    pub_year, pub_date,
                    w.get("cited_by_count", 0),
                    json.dumps(counts, ensure_ascii=False),
                    venue.get("id"), venue.get("display_name"), venue.get("type"),
                    w.get("type"), w.get("language"), oa,
                    json.dumps(insts, ensure_ascii=False),
                    json.dumps(w.get("concepts", []) or [], ensure_ascii=False)
                ])
                wrote_nodes += 1

                for cited in w.get("referenced_works", []) or []:
                    edges_writer.writerow([wid, cited, pub_year, pub_date])
                    wrote_edges += 1

                # mark processed
                processed_ids.add(wid)
                append_processed_id(PROCESSED_IDS_TXT, wid)

            nodes_fh.flush(); edges_fh.flush()

            total_collected += len(results)
            print_ckpt(
                f"year={year} — fetched {len(results)} (nodes+:{wrote_nodes}, edges+:{wrote_edges}) "
                f"total={total_collected:,} — next_cursor={'…' if next_cursor else None} — meta.count={count}"
            )

            # checkpoint after each page (safe resume without duplicating)
            if checkpoint_each_page:
                save_state({
                    "year": year if next_cursor else (year + 1),
                    "cursor": next_cursor if next_cursor else "*",
                    "total_collected": total_collected,
                    "start_year": start_year,
                    "end_year": end_year,
                    "stream_path": stream_path
                })

            if not next_cursor:
                year += 1
                cursor = "*"
                time.sleep(0.5 + random.uniform(0, 0.5))
            else:
                cursor = next_cursor
                polite_sleep(r, base=0.15, max_cap=30)

    finally:
        nodes_fh.close()
        edges_fh.close()

    print_ckpt(f"finished streaming — new works this run: {total_collected - grand_total_before:,}")
    print_ckpt(f"raw: {stream_path} — nodes: {NODES_CSV} — edges: {EDGES_CSV}")
    return stream_path, NODES_CSV, EDGES_CSV

# ---------- Optional: rollup + quick preview ----------
def rollup_and_preview(
    stream_jsonl=STREAM_JSONL,
    out_json=FINAL_JSON,
    out_csv_preview=FINAL_FLAT_CSV
):
    papers = []
    with open(stream_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            papers.append(json.loads(line))
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(papers, f, indent=2)

    # tiny preview table — keep concepts trimmed here only for readability
    rows = []
    for w in papers[:10000]:  # preview cap
        venue = ((w.get("primary_location") or {}).get("source")) or {}
        rows.append({
            "openalex_id": w.get("id"),
            "doi": w.get("doi"),
            "title": w.get("title"),
            "publication_year": w.get("publication_year"),
            "publication_date": w.get("publication_date"),
            "cited_by_count": w.get("cited_by_count", 0),
            "author_count": len(w.get("authorships", []) or []),
            "reference_count": len(w.get("referenced_works", []) or []),
            "venue": venue.get("display_name"),
            "venue_type": venue.get("type"),
            "is_oa": (w.get("open_access") or {}).get("is_oa", False),
            "type": w.get("type"),
            "language": w.get("language"),
            "concepts_top5": ", ".join([c.get("display_name","") for c in (w.get("concepts") or [])[:5]])
        })
    df = pd.DataFrame(rows)
    df.to_csv(out_csv_preview, index=False, encoding="utf-8")
    print_ckpt(f"rollup done → {out_json} — preview CSV → {out_csv_preview} ({len(df):,} rows)")
    return df

# ---------- Main ----------
if __name__ == "__main__":
    email = "david.dosu@cern.ch"
    stream_path, nodes_csv, edges_csv = get_quantum_papers_resumable(
        concept_id="C58053490",
        start_year=2000,
        end_year=2024,
        email=email,
        per_page=200,
        checkpoint_each_page=True
    )
    # optional one-shot rollup/preview
    rollup_and_preview(stream_path)


In [ ]:
# STEP 1: CANONICALIZE & QA
import os, json, math
import pandas as pd
import numpy as np

NODES_CSV = "qc_nodes.csv"
EDGES_CSV = "qc_edges.csv"
OUT_DIR   = "clean"

os.makedirs(OUT_DIR, exist_ok=True)

def to_quarter(dt: pd.Timestamp) -> str:
    if pd.isna(dt): return None
    return f"{dt.year}Q{((dt.month-1)//3)+1}"

def to_qindex(qstr: str) -> int:
    # numeric quarter index for arithmetic (e.g., 2000Q1 -> 2000*4+1)
    if qstr is None: return np.nan
    y, q = qstr.split("Q")
    return int(y)*4 + int(q)

# — load nodes
nodes = pd.read_csv(NODES_CSV, dtype=str, keep_default_na=False)
# enforce types
nodes["pub_year"]  = pd.to_numeric(nodes["pub_year"], errors="coerce")
nodes["pub_date"]  = pd.to_datetime(nodes["pub_date"], errors="coerce", utc=True).dt.tz_localize(None)
nodes["cited_by_count"] = pd.to_numeric(nodes["cited_by_count"], errors="coerce").fillna(0).astype(int)
nodes["is_oa"] = nodes["is_oa"].astype(str).str.upper().isin(["TRUE","1","T"])
nodes["qtr"] = nodes["pub_date"].apply(to_quarter)
nodes["qidx"] = nodes["qtr"].apply(to_qindex).astype("Int64")

# JSON columns to keep raw
json_cols = ["counts_by_year_json","institutions_json","concepts_json"]
for c in json_cols:
    nodes[c] = nodes[c].replace({"": "[]", "nan": "[]"})
    # leave as JSON strings; parse later only when needed

# keep only reasonable work types
valid_types = {
    "journal-article","proceedings-article","book-chapter","report","preprint","dataset"
}
nodes["type"] = nodes["type"].where(nodes["type"].isin(valid_types), other="other")

# drop rows with no date or quarter (can’t do time-based analysis)
nodes_clean = nodes[~nodes["qidx"].isna()].copy()

# — load edges
edges = pd.read_csv(EDGES_CSV, dtype=str, keep_default_na=False)
# fix types
edges["citing_pub_year"] = pd.to_numeric(edges["citing_pub_year"], errors="coerce")
edges["citing_pub_date"] = pd.to_datetime(edges["citing_pub_date"], errors="coerce", utc=True).dt.tz_localize(None)
edges = edges.dropna(subset=["citing_id","cited_id"])

# remove edges with missing citing date (no timeline → no upstream/downstream)
edges = edges[~edges["citing_pub_date"].isna()].copy()

# build quarter on edges (from citing work)
edges["citing_qtr"]  = edges["citing_pub_date"].apply(to_quarter)
edges["citing_qidx"] = edges["citing_qtr"].apply(to_qindex).astype("Int64")

# dedupe edges: multiple parsing passes can create dupes — keep one
edges_clean = edges.drop_duplicates(subset=["citing_id","cited_id"])

# keep only edges where citing_id is in nodes_clean
in_nodes = set(nodes_clean["openalex_id"])
edges_clean = edges_clean[edges_clean["citing_id"].isin(in_nodes)].copy()

# build references list per citing paper (set to avoid pair explosions from dupes)
refs = (
    edges_clean
    .groupby("citing_id")["cited_id"]
    .agg(lambda s: list(sorted(set(s))))
    .reset_index()
    .rename(columns={"cited_id":"ref_list"})
)

# basic coverage stats
total_nodes = len(nodes)
usable_nodes = len(nodes_clean)
total_edges = len(edges)
usable_edges = len(edges_clean)
print(f"Nodes: {usable_nodes}/{total_nodes} have valid dates and are usable.")
print(f"Edges: {usable_edges}/{total_edges} survive dedupe & date constraints.")
print(f"Mean refs / paper (usable): {refs['ref_list'].map(len).mean():.2f}")

# save Parquets for speed
nodes_clean.to_parquet(os.path.join(OUT_DIR, "nodes_clean.parquet"), index=False)
edges_clean.to_parquet(os.path.join(OUT_DIR, "edges_clean.parquet"), index=False)
refs.to_parquet(os.path.join(OUT_DIR, "paper_refs.parquet"), index=False)

# quick index for joins
nodes_idx = nodes_clean.set_index("openalex_id")[["pub_date","qidx","type","language","venue_id","venue_name","venue_type"]]
nodes_idx.to_parquet(os.path.join(OUT_DIR, "nodes_index.parquet"))
print("Wrote cleaned Parquets in ./clean/")


In [ ]:
# STEP 2: UPSTREAM NOVELTY (UNI) — ONLINE, LEAKAGE-FREE (ARRAY-SAFE)
import os, math, itertools, json
import pandas as pd
import numpy as np
from pandas.api.types import is_scalar

IN_DIR  = "clean"
OUT_DIR = "features"
os.makedirs(OUT_DIR, exist_ok=True)

nodes = pd.read_parquet(os.path.join(IN_DIR, "nodes_clean.parquet"))
refs  = pd.read_parquet(os.path.join(IN_DIR, "paper_refs.parquet"))

# join pub date + qidx onto refs and sort by time
nodes_min = nodes[["openalex_id","pub_date","qidx","type"]].rename(columns={"openalex_id":"citing_id"})
refs = refs.merge(nodes_min, on="citing_id", how="inner")
refs = refs.sort_values(["qidx","pub_date","citing_id"]).reset_index(drop=True)

# ---- normalize ref_list to a real Python list for every row
def as_ref_list(v):
    # None / NaN
    if v is None:
        return []
    if is_scalar(v):
        if pd.isna(v):
            return []
        if isinstance(v, str):
            s = v.strip()
            if s.startswith("[") and s.endswith("]"):
                try:
                    out = json.loads(s)
                    return out if isinstance(out, list) else []
                except Exception:
                    return []
            return []  # unknown scalar string
        # other scalars
        return []
    # list-like containers
    if isinstance(v, (list, tuple, set)):
        return list(v)
    # NumPy / pandas arrays
    if isinstance(v, (np.ndarray,)):
        return v.tolist()
    # last resort: try to iterate
    try:
        return list(v)
    except Exception:
        return []

refs["ref_list"] = refs["ref_list"].apply(as_ref_list)

# parameters
MAX_REFS_PER_PAPER = 120
EPS = 1e-12
USE_NPMI = True

# baseline counters (strictly “past only”)
n_papers = 0
single_counts = {}  # cited work -> count in earlier papers
pair_counts   = {}  # (a,b) with a<b  -> co-citation count in earlier papers

def npmi(pxy, px, py):
    if pxy <= 0 or px <= 0 or py <= 0:
        return -1.0
    num = math.log(pxy/(px*py) + EPS)
    den = -math.log(pxy + EPS)
    return num/den

rows = []
for _, row in refs.iterrows():
    wid   = row["citing_id"]
    qidx  = int(row["qidx"])
    t     = row["pub_date"]
    rlist = row["ref_list"]

    # unique & cap
    rset = list(dict.fromkeys(rlist))
    if len(rset) > MAX_REFS_PER_PAPER:
        rset = rset[:MAX_REFS_PER_PAPER]

    if n_papers == 0 or len(rset) < 2:
        rows.append({
            "openalex_id": wid,
            "pub_date": t,
            "qidx": qidx,
            "num_refs": len(rset),
            "num_pairs": 0,
            "share_unseen_pairs": np.nan if len(rset) < 2 else 1.0,
            "lift_mean": np.nan,
            "lift_p10": np.nan,
            "npmi_p10": np.nan,
            "npmi_mean": np.nan,
        })
        for r in set(rset):
            single_counts[r] = single_counts.get(r, 0) + 1
        n_papers += 1
        continue

    N = float(n_papers)
    scores_npmi = []
    scores_lift = []
    unseen = 0

    px_cache = {r: (single_counts.get(r, 0)/N) for r in rset}

    for a, b in itertools.combinations(sorted(rset), 2):
        key = (a,b) if a < b else (b,a)
        cxy = pair_counts.get(key, 0)
        if cxy == 0:
            unseen += 1
        pxy = cxy / N
        px = px_cache.get(a, 0.0)
        py = px_cache.get(b, 0.0)

        lift = pxy / max(px*py, EPS)
        scores_lift.append(lift)
        if USE_NPMI:
            scores_npmi.append(npmi(pxy, px, py))

    num_pairs = len(scores_lift)
    rows.append({
        "openalex_id": wid,
        "pub_date": t,
        "qidx": qidx,
        "num_refs": len(rset),
        "num_pairs": num_pairs,
        "share_unseen_pairs": (unseen / num_pairs) if num_pairs > 0 else np.nan,
        "lift_mean": float(np.mean(scores_lift)) if num_pairs else np.nan,
        "lift_p10":  float(np.percentile(scores_lift, 10)) if num_pairs else np.nan,
        "npmi_mean": float(np.mean(scores_npmi)) if scores_npmi else np.nan,
        "npmi_p10":  float(np.percentile(scores_npmi, 10)) if scores_npmi else np.nan,
    })

    # update baseline AFTER scoring
    for r in set(rset):
        single_counts[r] = single_counts.get(r, 0) + 1
    for a, b in itertools.combinations(sorted(rset), 2):
        key = (a,b) if a < b else (b,a)
        pair_counts[key] = pair_counts.get(key, 0) + 1

    n_papers += 1
    if n_papers % 10000 == 0:
        print(f"Processed {n_papers:,} papers — singles:{len(single_counts):,} pairs:{len(pair_counts):,}")

uni = pd.DataFrame(rows)

# sanity checks — MUST be > 0
assert (uni["num_refs"] > 0).any(), "All num_refs are zero — your paper_refs.parquet may be empty."
assert (uni["num_pairs"] > 0).any(), "All num_pairs are zero — pair formation failed."

uni.to_parquet(os.path.join(OUT_DIR, "upstream_uni.parquet"), index=False)
print("Wrote upstream features → features/upstream_uni.parquet")
print(uni[["num_refs","num_pairs","share_unseen_pairs","npmi_p10"]].describe())

In [ ]:
# Visualize upstream novelty metrics from features/upstream_uni.parquet
# — Produces 5 figures and saves them under ./plots/

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

FEATURES_PATH = "features/upstream_uni.parquet"
OUT_DIR = "plots"
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(FEATURES_PATH):
    raise FileNotFoundError(f"Expected file not found: {FEATURES_PATH}")

df = pd.read_parquet(FEATURES_PATH)

# — Basic cleaning / coercion
for col in ["num_refs", "num_pairs", "share_unseen_pairs", "npmi_p10", "qidx"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# — 1) Histogram: number of references per paper
plt.figure()
df["num_refs"].dropna().plot(kind="hist", bins=50)
plt.title("Distribution of References per Paper")
plt.xlabel("Number of references")
plt.ylabel("Count")
plt.tight_layout()
p1 = os.path.join(OUT_DIR, "upstream_hist_num_refs.png")
plt.savefig(p1, dpi=150)
plt.show()

# — 2) Histogram: share of unseen co-citation pairs
plt.figure()
df["share_unseen_pairs"].dropna().plot(kind="hist", bins=50)
plt.title("Distribution of Share of Unseen Co-citation Pairs")
plt.xlabel("Share of unseen pairs")
plt.ylabel("Count")
plt.tight_layout()
p2 = os.path.join(OUT_DIR, "upstream_hist_unseen_share.png")
plt.savefig(p2, dpi=150)
plt.show()

# — 3) Histogram: lower-tail nPMI (10th percentile)
plt.figure()
df["npmi_p10"].dropna().plot(kind="hist", bins=50)
plt.title("Distribution of Lower-tail nPMI (10th percentile)")
plt.xlabel("nPMI (10th percentile)")
plt.ylabel("Count")
plt.tight_layout()
p3 = os.path.join(OUT_DIR, "upstream_hist_npmi_p10.png")
plt.savefig(p3, dpi=150)
plt.show()

# — 4) Time trend: quarterly median of npmi_p10 (uses qidx if present)
trend_path = None
if "qidx" in df.columns:
    med_trend = (
        df.dropna(subset=["qidx", "npmi_p10"])
          .groupby("qidx")["npmi_p10"]
          .median()
          .reset_index()
          .sort_values("qidx")
    )
    if not med_trend.empty:
        plt.figure()
        plt.plot(med_trend["qidx"].values, med_trend["npmi_p10"].values)
        plt.title("Median Lower-tail nPMI (10th pct) by Quarter")
        plt.xlabel("Quarter index (year*4 + quarter)")
        plt.ylabel("Median nPMI (10th percentile)")
        plt.tight_layout()
        trend_path = os.path.join(OUT_DIR, "upstream_trend_npmi_p10.png")
        plt.savefig(trend_path, dpi=150)
        plt.show()

# — 5) Relationship: unseen share vs number of pairs (subsample for clarity)
scatter_path = None
sample = df.dropna(subset=["num_pairs", "share_unseen_pairs"])
if not sample.empty:
    n = min(20000, len(sample))
    sample = sample.sample(n, random_state=42)
    plt.figure()
    plt.scatter(sample["num_pairs"].values, sample["share_unseen_pairs"].values, s=5, alpha=0.5)
    plt.title("Unseen Share vs Number of Co-citation Pairs")
    plt.xlabel("Number of pairs (n(n−1)/2)")
    plt.ylabel("Share of unseen pairs")
    plt.tight_layout()
    scatter_path = os.path.join(OUT_DIR, "upstream_scatter_pairs_vs_unseen.png")
    plt.savefig(scatter_path, dpi=150)
    plt.show()

# — Quick textual summary
summary = {
    "n_rows": int(len(df)),
    "num_refs_mean": float(df["num_refs"].mean()),
    "num_pairs_mean": float(df["num_pairs"].mean()),
    "unseen_share_median": float(df["share_unseen_pairs"].median(skipna=True)),
    "npmi_p10_median": float(df["npmi_p10"].median(skipna=True)),
    "saved": {
        "refs_hist": p1,
        "unseen_hist": p2,
        "npmi_hist": p3,
        "trend_npmi": trend_path,
        "scatter_pairs_vs_unseen": scatter_path,
    }
}
print(summary)

In [ ]:
# STEP 2b — Upstream novelty with seen-only metrics + cap pressure
import os, math, itertools, json
import pandas as pd
import numpy as np
from pandas.api.types import is_scalar

IN_DIR  = "clean"
OUT_DIR = "features"
os.makedirs(OUT_DIR, exist_ok=True)

nodes = pd.read_parquet(os.path.join(IN_DIR, "nodes_clean.parquet"))
refs  = pd.read_parquet(os.path.join(IN_DIR, "paper_refs.parquet"))

nodes_min = nodes[["openalex_id","pub_date","qidx","type"]].rename(columns={"openalex_id":"citing_id"})
refs = refs.merge(nodes_min, on="citing_id", how="inner").sort_values(["qidx","pub_date","citing_id"]).reset_index(drop=True)

def as_ref_list(v):
    if v is None: return []
    if is_scalar(v):
        if isinstance(v, str):
            s = v.strip()
            if s.startswith("[") and s.endswith("]"):
                try:
                    x = json.loads(s)
                    return x if isinstance(x, list) else []
                except Exception:
                    return []
        try:
            return [] if pd.isna(v) else []
        except Exception:
            return []
    if isinstance(v, (list, tuple, set)): return list(v)
    if isinstance(v, np.ndarray): return v.tolist()
    try: return list(v)
    except Exception: return []

refs["ref_list"] = refs["ref_list"].apply(as_ref_list)

MAX_REFS_PER_PAPER = 120
EPS = 1e-12

def npmi(pxy, px, py):
    if pxy <= 0 or px <= 0 or py <= 0: return -1.0
    num = math.log(pxy/(px*py) + EPS)
    den = -math.log(pxy + EPS)
    return num/den

n_papers = 0
single_counts = {}
pair_counts   = {}

rows = []
for _, row in refs.iterrows():
    wid   = row["citing_id"]
    qidx  = int(row["qidx"])
    t     = row["pub_date"]
    rlist = row["ref_list"]

    rset = list(dict.fromkeys(rlist))
    cap_hit = int(len(rset) > MAX_REFS_PER_PAPER)
    if cap_hit: rset = rset[:MAX_REFS_PER_PAPER]

    if n_papers == 0 or len(rset) < 2:
        rows.append({
            "openalex_id": wid, "pub_date": t, "qidx": qidx,
            "num_refs": len(rset), "num_pairs": 0, "cap_hit": cap_hit,
            "share_unseen_pairs": np.nan if len(rset) < 2 else 1.0,
            "npmi_p10": np.nan, "npmi_mean": np.nan, "lift_p10": np.nan, "lift_mean": np.nan,
            "npmi_p10_seen": np.nan, "npmi_mean_seen": np.nan,
            "lift_p10_seen": np.nan, "lift_mean_seen": np.nan,
            "covered_pair_share": 0.0
        })
        for r in set(rset):
            single_counts[r] = single_counts.get(r, 0) + 1
        n_papers += 1
        continue

    N = float(n_papers)
    px_cache = {r: (single_counts.get(r, 0)/N) for r in rset}

    scores_npmi = []
    scores_lift = []
    scores_npmi_seen = []
    scores_lift_seen = []
    unseen = 0
    seen  = 0

    for a, b in itertools.combinations(sorted(rset), 2):
        key = (a,b) if a < b else (b,a)
        cxy = pair_counts.get(key, 0)
        pxy = cxy / N
        px = px_cache.get(a, 0.0)
        py = px_cache.get(b, 0.0)

        if cxy == 0:
            unseen += 1
        else:
            seen += 1

        lift = pxy / max(px*py, EPS)
        val_npmi = npmi(pxy, px, py)

        scores_lift.append(lift)
        scores_npmi.append(val_npmi)
        if cxy > 0:
            scores_lift_seen.append(lift)
            scores_npmi_seen.append(val_npmi)

    num_pairs = len(scores_lift)
    share_unseen = (unseen / num_pairs) if num_pairs > 0 else np.nan
    covered_share = 1.0 - share_unseen if num_pairs > 0 else 0.0

    def q10(x):
        return float(np.percentile(x, 10)) if x else np.nan
    def mean(x):
        return float(np.mean(x)) if x else np.nan

    rows.append({
        "openalex_id": wid, "pub_date": t, "qidx": qidx,
        "num_refs": len(rset), "num_pairs": num_pairs, "cap_hit": cap_hit,
        "share_unseen_pairs": share_unseen,
        "npmi_p10": q10(scores_npmi), "npmi_mean": mean(scores_npmi),
        "lift_p10": q10(scores_lift), "lift_mean": mean(scores_lift),
        "npmi_p10_seen": q10(scores_npmi_seen), "npmi_mean_seen": mean(scores_npmi_seen),
        "lift_p10_seen": q10(scores_lift_seen), "lift_mean_seen": mean(scores_lift_seen),
        "covered_pair_share": covered_share
    })

    for r in set(rset):
        single_counts[r] = single_counts.get(r, 0) + 1
    for a, b in itertools.combinations(sorted(rset), 2):
        key = (a,b) if a < b else (b,a)
        pair_counts[key] = pair_counts.get(key, 0) + 1

    n_papers += 1
    if n_papers % 10000 == 0:
        print(f"Processed {n_papers:,} papers — singles:{len(single_counts):,} pairs:{len(pair_counts):,}")

out = pd.DataFrame(rows)
out.to_parquet(os.path.join(OUT_DIR, "upstream_uni_v2.parquet"), index=False)
print("Wrote features/upstream_uni_v2.parquet")
print(out[["num_refs","num_pairs","cap_hit","share_unseen_pairs","covered_pair_share","npmi_p10_seen","npmi_mean_seen"]].describe())


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

uni = pd.read_parquet("features/upstream_uni_v2.parquet")

# Filter for coverage and refs
df = uni[(uni["num_refs"]>=5) & (uni["num_pairs"]>0) & (uni["covered_pair_share"]>=0.2)]

# 1. Correlation matrix
print(df[["share_unseen_pairs","npmi_p10_seen","npmi_mean_seen"]].corr(method="spearman"))

# 2. Cap effect
cap_means = df.groupby("cap_hit")[["share_unseen_pairs","npmi_p10_seen"]].mean()
print("\nCap effect (means):\n", cap_means)

# 3. Rolling median trend (post-burn-in qidx>=8050 ~ 2012+)
burned = df[df["qidx"]>=8050]
roll = (burned.sort_values("qidx")
               .groupby("qidx")[["share_unseen_pairs","npmi_p10_seen"]]
               .median()
               .rolling(8, min_periods=3).mean())
roll.plot(title="Rolling median novelty signals (burn-in applied)")
plt.show()

# 4. Placebo shuffle (random qidx)
shuffled = df.copy()
shuffled["qidx_shuf"] = np.random.permutation(shuffled["qidx"].values)
shuffled = shuffled.sort_values(["qidx_shuf","pub_date"])
# recompute distribution difference (simple version)
print("\nTrue npmi_p10_seen median:", df["npmi_p10_seen"].median())
print("Shuffled npmi_p10_seen median:", shuffled["npmi_p10_seen"].median())

In [ ]:
# SELF-HEAL + VALIDATE UPSTREAM SIGNALS
# - If features/upstream_uni_v2.parquet is missing, recompute it (Step 2b).
# - Then run validation: correlations, cap effect, rolling trends, split-half reliability.
import os, json, math, itertools, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.api.types import is_scalar

IN_CLEAN = "clean"
FEAT_DIR = "features"
PLOTS    = "plots/validation"
os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(PLOTS, exist_ok=True)

V2_PATH = os.path.join(FEAT_DIR, "upstream_uni_v2.parquet")

def as_ref_list(v):
    if v is None: return []
    if is_scalar(v):
        if isinstance(v, str):
            s = v.strip()
            if s.startswith("[") and s.endswith("]"):
                try:
                    x = json.loads(s)
                    return x if isinstance(x, list) else []
                except Exception:
                    return []
        try:
            return [] if pd.isna(v) else []
        except Exception:
            return []
    if isinstance(v, (list, tuple, set)): return list(v)
    if isinstance(v, np.ndarray): return v.tolist()
    try: return list(v)
    except Exception: return []

def npmi(pxy, px, py, eps=1e-12):
    if pxy <= 0 or px <= 0 or py <= 0: return -1.0
    num = math.log(pxy/(px*py) + eps)
    den = -math.log(pxy + eps)
    return num/den

def build_v2(max_refs=120):
    nodes = pd.read_parquet(os.path.join(IN_CLEAN, "nodes_clean.parquet"))
    refs  = pd.read_parquet(os.path.join(IN_CLEAN, "paper_refs.parquet"))
    nodes_min = nodes[["openalex_id","pub_date","qidx"]].rename(columns={"openalex_id":"citing_id"})
    refs = refs.merge(nodes_min, on="citing_id", how="inner").sort_values(["qidx","pub_date","citing_id"]).reset_index(drop=True)
    refs["ref_list"] = refs["ref_list"].apply(as_ref_list)

    n_papers = 0
    single_counts = {}   # cited -> count in earlier papers
    pair_counts   = {}   # (a,b) with a<b -> co-cite count in earlier papers
    rows = []

    for _, row in refs.iterrows():
        wid, qidx, t = row["citing_id"], int(row["qidx"]), row["pub_date"]
        rset = list(dict.fromkeys(row["ref_list"]))
        cap_hit = int(len(rset) > max_refs)
        if cap_hit: rset = rset[:max_refs]

        if n_papers == 0 or len(rset) < 2:
            rows.append({
                "openalex_id": wid, "pub_date": t, "qidx": qidx,
                "num_refs": len(rset), "num_pairs": 0, "cap_hit": cap_hit,
                "share_unseen_pairs": np.nan if len(rset) < 2 else 1.0,
                "npmi_p10": np.nan, "npmi_mean": np.nan, "lift_p10": np.nan, "lift_mean": np.nan,
                "npmi_p10_seen": np.nan, "npmi_mean_seen": np.nan,
                "lift_p10_seen": np.nan, "lift_mean_seen": np.nan,
                "covered_pair_share": 0.0
            })
            for r in set(rset):
                single_counts[r] = single_counts.get(r, 0) + 1
            n_papers += 1
            continue

        N = float(n_papers)
        px_cache = {r: (single_counts.get(r, 0)/N) for r in rset}

        scores_npmi = []; scores_lift = []
        scores_npmi_seen = []; scores_lift_seen = []
        unseen = seen = 0

        for a, b in itertools.combinations(sorted(rset), 2):
            key = (a,b) if a < b else (b,a)
            cxy = pair_counts.get(key, 0)
            pxy = cxy / N
            px, py = px_cache.get(a, 0.0), px_cache.get(b, 0.0)

            lift = pxy / max(px*py, 1e-12)
            v_npmi = npmi(pxy, px, py)

            scores_lift.append(lift); scores_npmi.append(v_npmi)
            if cxy == 0: unseen += 1
            else:
                seen += 1
                scores_lift_seen.append(lift); scores_npmi_seen.append(v_npmi)

        num_pairs = len(scores_lift)
        share_unseen = (unseen / num_pairs) if num_pairs else np.nan
        covered_share = 1.0 - share_unseen if num_pairs else 0.0
        q10 = (lambda x: float(np.percentile(x, 10)) if x else np.nan)
        mean = (lambda x: float(np.mean(x)) if x else np.nan)

        rows.append({
            "openalex_id": wid, "pub_date": t, "qidx": qidx,
            "num_refs": len(rset), "num_pairs": num_pairs, "cap_hit": cap_hit,
            "share_unseen_pairs": share_unseen,
            "npmi_p10": q10(scores_npmi), "npmi_mean": mean(scores_npmi),
            "lift_p10": q10(scores_lift), "lift_mean": mean(scores_lift),
            "npmi_p10_seen": q10(scores_npmi_seen), "npmi_mean_seen": mean(scores_npmi_seen),
            "lift_p10_seen": q10(scores_lift_seen), "lift_mean_seen": mean(scores_lift_seen),
            "covered_pair_share": covered_share
        })

        for r in set(rset):
            single_counts[r] = single_counts.get(r, 0) + 1
        for a, b in itertools.combinations(sorted(rset), 2):
            key = (a,b) if a < b else (b,a)
            pair_counts[key] = pair_counts.get(key, 0) + 1

        n_papers += 1
        if n_papers % 10000 == 0:
            print(f"Processed {n_papers:,} papers — singles:{len(single_counts):,} pairs:{len(pair_counts):,}")

    out = pd.DataFrame(rows)
    out.to_parquet(V2_PATH, index=False)
    return out

# — ensure v2 exists
if os.path.exists(V2_PATH):
    df = pd.read_parquet(V2_PATH)
else:
    print("upstream_uni_v2.parquet not found — building it now…")
    df = build_v2(max_refs=120)

# — VALIDATION

# Coerce types
for c in ["num_refs","num_pairs","share_unseen_pairs","npmi_p10_seen","npmi_mean_seen","covered_pair_share","qidx","cap_hit"]:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")

# Coverage & burn-in filters
df["seen_pairs"] = (df["num_pairs"] * df["covered_pair_share"]).fillna(0)
filtered = df[(df["num_refs"]>=5) & (df["num_pairs"]>0) & (df["seen_pairs"]>=20)]
burned   = filtered[filtered["qidx"] >= 8050]   # adjust threshold as you like

# 1) Correlations
corr = burned[["share_unseen_pairs","npmi_p10_seen","npmi_mean_seen"]].corr(method="spearman")
print("\nSpearman correlations:\n", corr)

# 2) Cap effect (effect size)
grp = burned.groupby((burned["num_refs"]==burned["num_refs"].max()).astype(int))
cap_means = grp[["share_unseen_pairs","npmi_p10_seen"]].mean()
cap_stds  = grp[["share_unseen_pairs","npmi_p10_seen"]].std()
n0, n1 = grp.size().get(0,0), grp.size().get(1,0)
print("\nCap effect (means):\n", cap_means)

def cohens_d(m0, s0, n0, m1, s1, n1):
    # pooled SD
    if n0<=1 or n1<=1: return np.nan
    sp = np.sqrt(((n0-1)*s0**2 + (n1-1)*s1**2) / (n0+n1-2))
    return (m1 - m0) / sp if sp>0 else np.nan

d_unseen = cohens_d(cap_means.loc[0,"share_unseen_pairs"], cap_stds.loc[0,"share_unseen_pairs"], n0,
                    cap_means.loc[1,"share_unseen_pairs"], cap_stds.loc[1,"share_unseen_pairs"], n1)
d_npmi   = cohens_d(cap_means.loc[0,"npmi_p10_seen"], cap_stds.loc[0,"npmi_p10_seen"], n0,
                    cap_means.loc[1,"npmi_p10_seen"], cap_stds.loc[1,"npmi_p10_seen"], n1)
print(f"\nCohen's d (cap vs no-cap): unseen={d_unseen:.3f}, npmi_p10_seen={d_npmi:.3f}")

# 3) Rolling medians over time
rm = (burned.sort_values("qidx").groupby("qidx")[["share_unseen_pairs","npmi_p10_seen"]]
      .median().rolling(8, min_periods=3).mean())

plt.figure()
plt.plot(rm.index.values, rm["share_unseen_pairs"].values)
plt.title("Rolling median — Share of unseen pairs (burn-in applied)")
plt.xlabel("Quarter index"); plt.ylabel("Median unseen share")
plt.tight_layout(); plt.savefig(os.path.join(PLOTS, "trend_unseen_share.png"), dpi=150); plt.show()

plt.figure()
plt.plot(rm.index.values, rm["npmi_p10_seen"].values)
plt.title("Rolling median — nPMI p10 (seen-only)")
plt.xlabel("Quarter index"); plt.ylabel("Median nPMI p10 (seen)")
plt.tight_layout(); plt.savefig(os.path.join(PLOTS, "trend_npmi_p10_seen.png"), dpi=150); plt.show()

# 4) Split-half reliability within quarters
def split_half_rel(dfq, col="npmi_p10_seen", seed=42):
    rnd = np.random.RandomState(seed)
    qs = sorted(dfq["qidx"].dropna().unique().tolist())
    m1, m2 = [], []
    for q in qs:
        sub = dfq[dfq["qidx"]==q][col].dropna().values
        if len(sub) < 6:  # need a few samples
            continue
        rnd.shuffle(sub)
        half = len(sub)//2
        m1.append(np.median(sub[:half])); m2.append(np.median(sub[half:]))
    if len(m1) >= 5:
        return float(pd.Series(m1).corr(pd.Series(m2), method="spearman")), len(m1)
    return np.nan, 0

r_npmi, k1 = split_half_rel(burned, "npmi_p10_seen")
r_unseen, k2 = split_half_rel(burned, "share_unseen_pairs")
print(f"\nSplit-half Spearman r: npmi_p10_seen={r_npmi:.3f} over {k1} quarters; unseen_share={r_unseen:.3f} over {k2} quarters")

print("\nValidation done. Plots saved to:", PLOTS)


In [ ]:
# === Year-based, leakage-free pipeline (Upstream + Downstream) ===
# - Baseline uses only prior YEARS (no within-year peeking).
# - Upstream features -> features/upstream_uni_year.parquet
# - Downstream outcomes (H_years=2) -> features/downstream_Hy2.parquet
# - Deterministic ref sampling for cap fairness.

import os, json, math, itertools, hashlib, random, collections
import numpy as np
import pandas as pd

CLEAN_DIR = "clean"
FEAT_DIR  = "features"
os.makedirs(FEAT_DIR, exist_ok=True)

# ---------- params ----------
MAX_REFS = 120                   # bibliography cap (after dedupe)
USE_DETERMINISTIC_CAP = True     # sample refs instead of "first 120"
H_YEARS = 2                      # downstream horizon in YEARS (≈ 8 quarters)
EPS = 1e-12                      # numerical smoothing

# ---------- io ----------
nodes_path = os.path.join(CLEAN_DIR, "nodes_clean.parquet")
refs_path  = os.path.join(CLEAN_DIR, "paper_refs.parquet")
edges_path = os.path.join(CLEAN_DIR, "edges_clean.parquet")
assert os.path.exists(nodes_path) and os.path.exists(refs_path), "Missing clean/{nodes, paper_refs}.parquet"

nodes = pd.read_parquet(nodes_path)
# ensure pub_year exists
if "pub_year" not in nodes.columns:
    # derive from pub_date if needed
    if "pub_date" in nodes.columns:
        nodes["pub_year"] = pd.to_datetime(nodes["pub_date"], errors="coerce").dt.year
    else:
        raise ValueError("nodes_clean.parquet lacks pub_year and pub_date; cannot build year-based chronology.")

nodes["pub_year"] = pd.to_numeric(nodes["pub_year"], errors="coerce")
nodes = nodes.dropna(subset=["pub_year"])
nodes["pub_year"] = nodes["pub_year"].astype(int)

# minimal metadata index (used later for downstream)
meta_cols = ["title","venue_name","venue_type","institutions_json","pub_year"]
keep_meta = [c for c in meta_cols if c in nodes.columns]
nodes_meta = nodes.set_index("openalex_id")[keep_meta] if keep_meta else nodes.set_index("openalex_id")[[]]

# ---------- load refs; attach year ----------
refs = pd.read_parquet(refs_path)
if "citing_id" not in refs.columns:
    raise ValueError("paper_refs.parquet must have column 'citing_id' and 'ref_list'.")

refs = refs.merge(nodes[["openalex_id","pub_year"]].rename(columns={"openalex_id":"citing_id"}),
                  on="citing_id", how="left")
refs = refs.dropna(subset=["pub_year"])
refs["pub_year"] = refs["pub_year"].astype(int)

# normalize ref_list safely
def as_ref_list(v):
    if v is None: return []
    if isinstance(v, list): return v
    if isinstance(v, (set, tuple)): return list(v)
    if isinstance(v, np.ndarray): return v.tolist()
    if isinstance(v, str):
        s = v.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                x = json.loads(s)
                return x if isinstance(x, list) else []
            except Exception:
                return []
        return []
    try:
        return list(v)
    except Exception:
        return []

refs["ref_list"] = refs["ref_list"].apply(as_ref_list)

# deterministic sampling for cap fairness
def det_sample_refs(refs_list, k, seed_key):
    if len(refs_list) <= k:
        return refs_list
    # stable order + deterministic RNG from paper id
    uniq = list(dict.fromkeys(refs_list))
    rnd = random.Random(int(hashlib.blake2b(str(seed_key).encode("utf-8"), digest_size=8).hexdigest(), 16))
    if len(uniq) <= k:
        return uniq
    idx = list(range(len(uniq)))
    rnd.shuffle(idx)
    take = sorted(idx[:k])
    return [uniq[i] for i in take]

# ---------- upstream: year-based leakage-free scoring ----------
# Strategy:
#   For year Y in ascending order:
#       score all papers with pub_year == Y using BASELINE from years < Y
#       then update baseline counts with all Y papers (singles + pairs)
#
#   This eliminates dependence on intra-year ordering.

single_counts = collections.Counter()  # work_id -> # earlier-year papers that cited it
pair_counts   = collections.Counter()  # (a,b) with a<b -> # earlier-year papers that co-cited them
baseline_N    = 0                      # number of papers in baseline (prior years)

rows = []
years = sorted(refs["pub_year"].unique())
print(f"Scoring upstream by year across {len(years)} years: {years[0]}–{years[-1]}")

for Y in years:
    year_df = refs[refs["pub_year"] == Y].copy()
    # ---- score using baseline from < Y ----
    for _, r in year_df.iterrows():
        wid = r["citing_id"]
        y   = int(r["pub_year"])
        rlist = r["ref_list"]
        # dedupe, then cap
        runiq = list(dict.fromkeys(rlist))
        cap_hit = False
        if USE_DETERMINISTIC_CAP:
            if len(runiq) > MAX_REFS:
                cap_hit = True
                runiq = det_sample_refs(runiq, MAX_REFS, seed_key=wid)
        else:
            if len(runiq) > MAX_REFS:
                cap_hit = True
                runiq = runiq[:MAX_REFS]

        num_refs = len(runiq)
        if num_refs < 2 or baseline_N == 0:
            rows.append({
                "openalex_id": wid,
                "pub_year": y,
                "num_refs": num_refs,
                "num_pairs": 0 if num_refs < 2 else num_refs*(num_refs-1)//2,
                "cap_hit": int(cap_hit),
                "share_unseen_pairs": np.nan if baseline_N==0 else 1.0,  # if baseline empty, everything is "new" but we mark NaN to be explicit
                "covered_pair_share": np.nan if baseline_N==0 else 0.0,
                "seen_pairs": 0,
                "npmi_p10_seen": np.nan,
                "npmi_mean_seen": np.nan,
                "lift_mean_seen": np.nan,
            })
            continue

        # pre-cache singles probability from baseline
        N = float(baseline_N)
        px_cache = {w: (single_counts[w]/N) for w in runiq}
        unseen = 0
        npmi_seen_vals = []
        lift_seen_vals = []

        # compute pairs
        pairs = 0
        for a, b in itertools.combinations(sorted(runiq), 2):
            pairs += 1
            key = (a,b) if a < b else (b,a)
            cxy = pair_counts.get(key, 0)
            if cxy == 0:
                unseen += 1
                continue  # unseen excluded from seen-only tails
            pxy = cxy / N
            px  = px_cache.get(a, 0.0)
            py  = px_cache.get(b, 0.0)
            # lift
            lift = pxy / max(px*py, EPS)
            lift_seen_vals.append(lift)
            # nPMI
            if pxy > 0 and px > 0 and py > 0:
                num = math.log(pxy/(px*py) + EPS)
                den = -math.log(pxy + EPS)
                npmi_seen_vals.append(num/den)

        num_pairs = pairs
        seen_pairs = num_pairs - unseen
        share_unseen = (unseen / num_pairs) if num_pairs > 0 else np.nan
        covered_share = 1.0 - share_unseen if num_pairs > 0 else np.nan

        rows.append({
            "openalex_id": wid,
            "pub_year": y,
            "num_refs": num_refs,
            "num_pairs": num_pairs,
            "cap_hit": int(cap_hit),
            "share_unseen_pairs": share_unseen,
            "covered_pair_share": covered_share,
            "seen_pairs": int(seen_pairs),
            "npmi_p10_seen": float(np.percentile(npmi_seen_vals, 10)) if npmi_seen_vals else np.nan,
            "npmi_mean_seen": float(np.mean(npmi_seen_vals)) if npmi_seen_vals else np.nan,
            "lift_mean_seen": float(np.mean(lift_seen_vals)) if lift_seen_vals else np.nan,
        })

    # ---- after scoring all Y-papers, UPDATE baseline with their singles & pairs ----
    added = 0
    for _, r in year_df.iterrows():
        runiq = list(dict.fromkeys(r["ref_list"]))
        if USE_DETERMINISTIC_CAP and len(runiq) > MAX_REFS:
            runiq = det_sample_refs(runiq, MAX_REFS, seed_key=r["citing_id"])
        elif len(runiq) > MAX_REFS:
            runiq = runiq[:MAX_REFS]

        # singles
        for c in set(runiq):
            single_counts[c] += 1
        # pairs
        for a, b in itertools.combinations(sorted(runiq), 2):
            key = (a,b) if a < b else (b,a)
            pair_counts[key] += 1
        added += 1
    baseline_N += added
    print(f"Year {Y}: scored {len(year_df):,} papers; baseline_N now {baseline_N:,} (singles {len(single_counts):,}, pairs {len(pair_counts):,})")

up_year = pd.DataFrame(rows)
up_year.to_parquet(os.path.join(FEAT_DIR, "upstream_uni_year.parquet"), index=False)
print("Wrote features/upstream_uni_year.parquet")
print(up_year[["share_unseen_pairs","npmi_p10_seen","covered_pair_share","seen_pairs","num_refs","cap_hit"]].describe())

# ---------- downstream by YEAR (H_YEARS) ----------
if os.path.exists(edges_path):
    edges = pd.read_parquet(edges_path)[["cited_id","citing_id"]]
    # join years for citing and cited
    cited_year = nodes.set_index("openalex_id")["pub_year"]
    citing_year = cited_year  # same index
    edges["cited_year"]  = edges["cited_id"].map(cited_year)
    edges["citing_year"] = edges["citing_id"].map(citing_year)
    edges = edges.dropna(subset=["cited_year","citing_year"])
    edges["cited_year"]  = edges["cited_year"].astype(int)
    edges["citing_year"] = edges["citing_year"].astype(int)
    edges["dt_year"] = edges["citing_year"] - edges["cited_year"]

    # group citations within (0, H_YEARS]
    H = int(H_YEARS)
    edges_H = edges[(edges["dt_year"] > 0) & (edges["dt_year"] <= H)].copy()

    # helper: normalized entropy
    def entropy_norm(counts):
        tot = sum(counts)
        if tot <= 0: return np.nan
        p = [c/tot for c in counts if c>0]
        if len(p) <= 1: return 0.0
        Hs = -sum(pi*math.log(pi) for pi in p)
        return float(Hs / math.log(len(p)))

    # build downstream rows per focal
    ds_rows = []
    grp = edges_H.groupby("cited_id")
    for wid, g in grp:
        fy = int(cited_year.get(wid, np.nan))
        if not isinstance(fy, (int, np.integer)) and not (isinstance(fy, float) and not np.isnan(fy)):
            continue
        # per-year series within horizon
        per_y = g["dt_year"].value_counts().reindex(range(1, H+1), fill_value=0).sort_index()
        x = np.arange(1, H+1); y = per_y.values.astype(float)
        slope = float(np.polyfit(x, y, 1)[0]) if y.sum()>0 else 0.0
        dy = np.diff(y, prepend=0)
        accel = float(np.polyfit(x, dy, 1)[0]) if len(y) >= 3 and dy.std()>0 else 0.0

        # citing attributes for diffusion
        citing_ids = g["citing_id"].tolist()
        meta = nodes_meta.loc[nodes_meta.index.intersection(citing_ids)]
        vt_counts = meta["venue_type"].fillna("NA").value_counts().to_dict() if "venue_type" in meta.columns else {}
        v_counts  = meta["venue_name"].fillna("NA").value_counts().to_dict() if "venue_name" in meta.columns else {}
        # country codes from institutions_json (list of dicts)
        cc_counter = collections.Counter()
        if "institutions_json" in meta.columns:
            for js in meta["institutions_json"]:
                try:
                    arr = json.loads(js) if isinstance(js, str) else (js or [])
                except Exception:
                    arr = []
                for i in arr:
                    if isinstance(i, dict):
                        cc = (i.get("country_code") or "").strip().upper()
                        if cc: cc_counter[cc] += 1

        ds_rows.append({
            "openalex_id": wid,
            "focal_year": int(fy),
            "cits_Hy": int(len(g)),
            "pery_slope": slope,
            "pery_accel": accel,
            "venue_type_entropy": entropy_norm(list(vt_counts.values())),
            "venue_name_entropy": entropy_norm(list(v_counts.values())),
            "country_entropy": entropy_norm(list(cc_counter.values())),
            "unique_venues": int(len(v_counts)),
            "unique_countries": int(len(cc_counter)),
        })

    down_year = pd.DataFrame(ds_rows)
    down_year.to_parquet(os.path.join(FEAT_DIR, f"downstream_Hy{H}.parquet"), index=False)
    print(f"Wrote features/downstream_Hy{H}.parquet (H={H} years)")
    print(down_year[["cits_Hy","pery_slope","country_entropy","venue_type_entropy"]].describe())
else:
    print("No clean/edges_clean.parquet found — skipping downstream-by-year.")


In [ ]:
# === Upstream plots by publication YEAR ===
# Reads features/upstream_uni_year.parquet (year-based, leakage-free upstream features)
# Saves plots into ./plots_year/

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FEAT_DIR = "features"
OUT_DIR  = "plots_year"
os.makedirs(OUT_DIR, exist_ok=True)

path = os.path.join(FEAT_DIR, "upstream_uni_year.parquet")
assert os.path.exists(path), "Missing features/upstream_uni_year.parquet — run the year-based pipeline first."

u = pd.read_parquet(path)

# Coerce needed columns
u["pub_year"] = pd.to_numeric(u["pub_year"], errors="coerce").astype("Int64")
u["share_unseen_pairs"] = pd.to_numeric(u["share_unseen_pairs"], errors="coerce")
# seen-only tail; if absent, fall back to npmi_p10
tail_col = "npmi_p10_seen" if "npmi_p10_seen" in u.columns else "npmi_p10"
u[tail_col] = pd.to_numeric(u[tail_col], errors="coerce")
u["num_refs"] = pd.to_numeric(u.get("num_refs"), errors="coerce")
# coverage direct or derived
if "covered_pair_share" in u.columns:
    u["coverage"] = pd.to_numeric(u["covered_pair_share"], errors="coerce")
else:
    u["coverage"] = 1.0 - u["share_unseen_pairs"]

# 1) Histogram — Share of NEW pairs
plt.figure()
u["share_unseen_pairs"].dropna().plot(kind="hist", bins=50)
plt.title("Distribution of Share of NEW Co-citation Pairs (by paper)")
plt.xlabel("Share of NEW pairs (never co-cited before)")
plt.ylabel("Number of papers")
plt.tight_layout()
p1 = os.path.join(OUT_DIR, "hist_share_new_pairs_year.png")
plt.savefig(p1, dpi=150); plt.close()

# 2) Histogram — nPMI(10%) among SEEN pairs
plt.figure()
u[tail_col].dropna().plot(kind="hist", bins=50)
label_tail = "nPMI (10th percentile) — SEEN pairs only" if tail_col.endswith("_seen") else "nPMI (10th percentile)"
plt.title("Distribution of Lower-tail nPMI")
plt.xlabel(label_tail)
plt.ylabel("Number of papers")
plt.tight_layout()
p2 = os.path.join(OUT_DIR, "hist_npmi_tail_seen_year.png")
plt.savefig(p2, dpi=150); plt.close()

# 3) Trend — Share of papers hitting the 120-ref cap by YEAR
cap_val = int(u["num_refs"].max()) if u["num_refs"].notna().any() else 120
df_cap = u.dropna(subset=["pub_year"]).copy()
df_cap["at_cap"] = (df_cap["num_refs"] == cap_val).astype(int)
cap_by_y = df_cap.groupby("pub_year")["at_cap"].mean().reset_index()

plt.figure()
plt.plot(cap_by_y["pub_year"], cap_by_y["at_cap"])
plt.title(f"Share of Papers Hitting the {cap_val}-Reference Cap by Year")
plt.xlabel("Publication year")
plt.ylabel("Share at cap")
plt.tight_layout()
p3 = os.path.join(OUT_DIR, "trend_cap_pressure_year.png")
plt.savefig(p3, dpi=150); plt.close()

# 4) Hexbin — NEW-share vs nPMI(10% seen) with optional thresholds (year-agnostic)
df_hex = u.dropna(subset=["share_unseen_pairs", tail_col]).copy()
plt.figure()
plt.hexbin(df_hex["share_unseen_pairs"], df_hex[tail_col], gridsize=45, mincnt=10)
plt.title("New recombinations vs. anchored novelty (year-based upstream)")
plt.xlabel("Share of NEW co-citation pairs in this paper")
ylabel = "nPMI (10th percentile) among PREVIOUSLY co-cited pairs" if tail_col.endswith("_seen") else "nPMI (10th percentile)"
plt.ylabel(ylabel)

# Optional: show thresholds you’ve been using
X_THRESH = 0.70
Y_THRESH = 0.20
plt.axvline(X_THRESH)
plt.axhline(Y_THRESH)

plt.tight_layout()
p4 = os.path.join(OUT_DIR, "hex_new_vs_tail_seen_year.png")
plt.savefig(p4, dpi=150); plt.close()

# 5) (Optional) Coverage trend by YEAR (if you want it)
cov_y = (u.dropna(subset=["pub_year","coverage"])
           .groupby("pub_year")["coverage"].median()
           .reset_index())
if not cov_y.empty:
    plt.figure()
    plt.plot(cov_y["pub_year"], cov_y["coverage"])
    plt.title("Median Coverage by Publication Year")
    plt.xlabel("Publication year")
    plt.ylabel("Coverage = fraction of pairs already known")
    plt.tight_layout()
    p5 = os.path.join(OUT_DIR, "trend_coverage_year.png")
    plt.savefig(p5, dpi=150); plt.close()
else:
    p5 = None

print("Saved:")
for p in [p1,p2,p3,p4,p5]:
    if p: print(" -", p)


In [ ]:
# View upstream-year plots in Colab
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

PLOT_DIR = "plots_year"  # change if you saved elsewhere
files = [
    "hist_share_new_pairs_year.png",
    "hist_npmi_tail_seen_year.png",
    "trend_cap_pressure_year.png",
    "hex_new_vs_tail_seen_year.png",
    "trend_coverage_year.png",
]

paths = [os.path.join(PLOT_DIR, f) for f in files]

for p in paths:
    if not os.path.exists(p):
        print(f"[skip] Not found: {p}")
        continue
    img = mpimg.imread(p)
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(os.path.basename(p), pad=10)
    plt.show()


In [ ]:
# Optional: data-driven crosshair (e.g., 75th percentiles with coverage floor)
# Load the data first if running this cell independently
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

FEAT_PATH = "features/upstream_uni_year.parquet"   # year-based upstream features
assert os.path.exists(FEAT_PATH), "Missing features/upstream_uni_year.parquet. Run the year-based upstream pipeline."
u = pd.read_parquet(FEAT_PATH)

SEEN_MIN = 20
tail_col = "npmi_p10_seen" if "npmi_p10_seen" in u.columns else "npmi_p10"

df = u.dropna(subset=["share_unseen_pairs", tail_col]).copy()
df = df[df.get("seen_pairs", 0).fillna(0) >= SEEN_MIN]

X_THRESH = float(df["share_unseen_pairs"].quantile(0.75))  # top quartile of NEW-share
Y_THRESH = float(df[tail_col].quantile(0.75))              # top quartile of tail nPMI

plt.axvline(X_THRESH)
plt.axhline(Y_THRESH)
print(f"Crosshair set to 75th pct: X={X_THRESH:.3f}, Y={Y_THRESH:.3f} (seen_pairs ≥ {SEEN_MIN})")

In [ ]:
# === List all papers that exceeded 120 references ===
# Inputs:
#   clean/paper_refs.parquet          (citing_id, ref_list)
#   clean/nodes_clean.parquet         (openalex_id, title, venue_name, venue_type, pub_year, qidx, etc.)
# Optional:
#   features/upstream_uni_v2.parquet  (to compare capped num_refs & flag cap-hit in features)
#
# Output:
#   reports/papers_over_120_refs.csv

import os, json
import numpy as np
import pandas as pd

CLEAN_DIR = "clean"
FEAT_DIR  = "features"
OUT_DIR   = "reports"
os.makedirs(OUT_DIR, exist_ok=True)

refs_path  = os.path.join(CLEAN_DIR, "paper_refs.parquet")
nodes_path = os.path.join(CLEAN_DIR, "nodes_clean.parquet")
feat_path  = os.path.join(FEAT_DIR, "upstream_uni_v2.parquet")  # optional

assert os.path.exists(refs_path),  f"Missing {refs_path}"
assert os.path.exists(nodes_path), f"Missing {nodes_path}"

# --- Safe parser for ref_list ---
def as_ref_list(v):
    if v is None: return []
    if isinstance(v, list): return v
    if isinstance(v, (set, tuple)): return list(v)
    if isinstance(v, np.ndarray): return v.tolist()
    if isinstance(v, str):
        s = v.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                x = json.loads(s)
                return x if isinstance(x, list) else []
            except Exception:
                return []
        return []
    try:
        return list(v)
    except Exception:
        return []

# --- Load & compute raw ref counts (deduped) ---
refs = pd.read_parquet(refs_path)
refs["ref_list"] = refs["ref_list"].apply(as_ref_list)
# dedupe while preserving order
refs["num_refs_raw"] = refs["ref_list"].apply(lambda L: len(dict.fromkeys(L)))

# --- Join metadata ---
meta_cols = ["openalex_id","title","venue_name","venue_type","pub_year","qidx","doi","type"]
nodes = pd.read_parquet(nodes_path)
nodes = nodes[[c for c in meta_cols if c in nodes.columns]]
df = refs.merge(nodes.rename(columns={"openalex_id":"citing_id"}), on="citing_id", how="left")

# --- Optional: compare to capped counts from upstream features (to see truncation) ---
if os.path.exists(feat_path):
    up = pd.read_parquet(feat_path, columns=["openalex_id","num_refs"]).rename(
        columns={"openalex_id":"citing_id","num_refs":"num_refs_capped"})
    df = df.merge(up, on="citing_id", how="left")
else:
    df["num_refs_capped"] = np.nan

# --- Filter: strictly more than 120 refs (raw) ---
over = df[df["num_refs_raw"] > 120].copy()
over.sort_values("num_refs_raw", ascending=False, inplace=True)

# --- Add a few useful flags/columns ---
cap_val = 120
over["cap_hit_expected"] = True  # by definition raw>120 means it *should* be capped in upstream
if over["num_refs_capped"].notna().any():
    over["cap_applied_in_features"] = (over["num_refs_capped"] == cap_val)

# Reorder columns for readability
cols_order = [
    "citing_id","title","pub_year","qidx","venue_name","venue_type","doi","type",
    "num_refs_raw","num_refs_capped","cap_applied_in_features","cap_hit_expected"
]
cols_order = [c for c in cols_order if c in over.columns]
over = over[cols_order]

# --- Save & preview ---
out_csv = os.path.join(OUT_DIR, "papers_over_120_refs.csv")
over.to_csv(out_csv, index=False)

print(f"Total papers with >120 unique references: {len(over):,}")
print(f"Saved full list -> {out_csv}")

# Show a quick preview
print("\nTop 20 by raw ref count:")
print(over.head(20).to_string(index=False))


In [ ]:
# Fraction of papers with >= 120 references (deduped)
# Inputs:
#   clean/paper_refs.parquet        (columns: citing_id, ref_list)
# Optional:
#   features/upstream_uni_year.parquet (to confirm cap value & compare)

import os, json
import numpy as np
import pandas as pd

REFS_PATH = "clean/paper_refs.parquet"
FEAT_PATH = "features/upstream_uni_year.parquet"  # optional

assert os.path.exists(REFS_PATH), f"Missing {REFS_PATH}"

# ---- safe parser for ref_list ----
def as_ref_list(v):
    if v is None: return []
    if isinstance(v, list): return v
    if isinstance(v, (set, tuple)): return list(v)
    if isinstance(v, np.ndarray): return v.tolist()
    if isinstance(v, str):
        s = v.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                x = json.loads(s)
                return x if isinstance(x, list) else []
            except Exception:
                return []
        return []
    try:
        return list(v)
    except Exception:
        return []

# ---- load & compute raw (deduped) counts ----
refs = pd.read_parquet(REFS_PATH)
refs["ref_list"] = refs["ref_list"].apply(as_ref_list)
# dedupe while preserving order
refs["num_refs_raw"] = refs["ref_list"].apply(lambda L: len(dict.fromkeys(L)))

TOTAL = len(refs)
AT_LEAST_120 = (refs["num_refs_raw"] >= 120).sum()
FRACTION = AT_LEAST_120 / TOTAL if TOTAL else np.nan

print(f"Total papers processed: {TOTAL:,}")
print(f"Papers with ≥120 unique references: {AT_LEAST_120:,} "
      f"({FRACTION:.3%})")

# ---- per-year breakdown (if pub_year available) ----
if "pub_year" in refs.columns and pd.api.types.is_integer_dtype(refs["pub_year"]):
    by = (refs.assign(ge120 = refs["num_refs_raw"] >= 120)
               .groupby("pub_year")
               .agg(n_total=("citing_id","size"),
                    n_ge120=("ge120","sum"))
               .reset_index())
    by["share_ge120"] = by["n_ge120"] / by["n_total"]
    print("\nPer-year share (last 10 rows):")
    print(by.tail(10).to_string(index=False))
else:
    print("\nNo integer pub_year in paper_refs.parquet; skipping per-year breakdown.")

# ---- optional: compare to capped features (confirm cap value) ----
if os.path.exists(FEAT_PATH):
    up = pd.read_parquet(FEAT_PATH, columns=["openalex_id","num_refs"]) \
            .rename(columns={"openalex_id":"citing_id","num_refs":"num_refs_capped"})
    chk = refs.merge(up, on="citing_id", how="left")
    cap_val = int(chk["num_refs_capped"].max()) if chk["num_refs_capped"].notna().any() else 120
    truncated = (chk["num_refs_raw"] > cap_val).sum()
    print(f"\nUsing cap value {cap_val}. Papers truncated by cap in features: {truncated:,}")


In [ ]:
# === Distribution per year for "novel + anchored" papers ===
# Region: share_new_pairs ≥ X_THRESH  AND  npmi_p10_seen ≥ Y_THRESH
# Guardrail: seen_pairs ≥ SEEN_MIN
#
# Input:  features/upstream_uni_year.parquet  (from the year-based pipeline)
# Output: reports/flagged_by_year.csv  +  two plots in plots_year/

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FEAT_PATH = "features/upstream_uni_year.parquet"
OUT_CSV   = "reports/flagged_by_year.csv"
PLOT_DIR  = "plots_year"
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

assert os.path.exists(FEAT_PATH), "Run the year-based upstream pipeline first (upstream_uni_year.parquet)"

# --- thresholds & guardrails ---
X_THRESH = 0.70   # min share of NEW pairs
Y_THRESH = 0.20   # min nPMI tail among seen pairs
SEEN_MIN = 20     # minimum number of seen pairs for stability

# --- load & coerce ---
u = pd.read_parquet(FEAT_PATH)

# choose tail column (prefer seen-only)
tail_col = "npmi_p10_seen" if "npmi_p10_seen" in u.columns else ("npmi_p10" if "npmi_p10" in u.columns else None)
if tail_col is None:
    raise ValueError("No nPMI tail column found. Expected 'npmi_p10_seen' or 'npmi_p10' in upstream_uni_year.parquet.")

u["pub_year"] = pd.to_numeric(u["pub_year"], errors="coerce").astype("Int64")
u["share_unseen_pairs"] = pd.to_numeric(u.get("share_unseen_pairs"), errors="coerce")
u[tail_col] = pd.to_numeric(u[tail_col], errors="coerce")
u["seen_pairs"] = pd.to_numeric(u.get("seen_pairs"), errors="coerce")

# sanity: drop rows missing the essentials
df = u.dropna(subset=["pub_year","share_unseen_pairs",tail_col]).copy()

# --- define the region & guardrail ---
df["is_flagged"] = (
    (df["share_unseen_pairs"] >= X_THRESH) &
    (df[tail_col] >= Y_THRESH) &
    (df["seen_pairs"].fillna(0) >= SEEN_MIN)
)

# --- yearly distribution (counts and shares) ---
year_grp = df.groupby("pub_year", dropna=True)
out = pd.DataFrame({
    "n_total": year_grp.size(),
    "n_flagged": year_grp["is_flagged"].sum()
}).reset_index()
out["share_flagged"] = out["n_flagged"] / out["n_total"]

# (optional diagnostics)
out["median_share_new"] = year_grp["share_unseen_pairs"].median().values
out[f"median_{tail_col}"] = year_grp[tail_col].median().values
out["median_seen_pairs"] = year_grp["seen_pairs"].median().values

# save CSV
out.to_csv(OUT_CSV, index=False)
print(f"Saved per-year distribution -> {OUT_CSV}")
print(out.tail(10))

# --- plots ---
# 1) Count of flagged papers per year
plt.figure()
plt.plot(out["pub_year"], out["n_flagged"])
plt.title(f'Count of "novel + anchored" papers per year\n'
          f'(share_new ≥ {X_THRESH}, {tail_col} ≥ {Y_THRESH}, seen_pairs ≥ {SEEN_MIN})')
plt.xlabel("Publication year"); plt.ylabel("Count")
plt.tight_layout()
p1 = os.path.join(PLOT_DIR, "flagged_count_per_year.png")
plt.savefig(p1, dpi=150); plt.show()

# 2) Share of flagged papers per year
plt.figure()
plt.plot(out["pub_year"], out["share_flagged"])
plt.title(f'Share of "novel + anchored" papers per year')
plt.xlabel("Publication year"); plt.ylabel("Share of papers")
plt.tight_layout()
p2 = os.path.join(PLOT_DIR, "flagged_share_per_year.png")
plt.savefig(p2, dpi=150); plt.show()

print("Saved plots:")
print(" -", p1)
print(" -", p2)


In [ ]:
# Sensitivity on coverage floor: seen_pairs ≥ {10, 20, 30}
# Reads: features/upstream_uni_year.parquet
# Writes: reports/flagged_by_year_seenmin{K}.csv
#         plots_year/flagged_count_per_year_seenmin{K}.png
#         plots_year/flagged_share_per_year_seenmin{K}.png

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FEAT_PATH = "features/upstream_uni_year.parquet"
assert os.path.exists(FEAT_PATH), "Missing features/upstream_uni_year.parquet"

u = pd.read_parquet(FEAT_PATH)
tail_col = "npmi_p10_seen" if "npmi_p10_seen" in u.columns else "npmi_p10"

u["pub_year"] = pd.to_numeric(u["pub_year"], errors="coerce").astype("Int64")
u["share_unseen_pairs"] = pd.to_numeric(u["share_unseen_pairs"], errors="coerce")
u[tail_col] = pd.to_numeric(u[tail_col], errors="coerce")
u["seen_pairs"] = pd.to_numeric(u.get("seen_pairs"), errors="coerce")

X_THRESH = 0.70
Y_THRESH = 0.20

os.makedirs("reports", exist_ok=True)
os.makedirs("plots_year", exist_ok=True)

for SEEN_MIN in (10, 20, 30):
    df = u.dropna(subset=["pub_year","share_unseen_pairs",tail_col]).copy()
    df["is_flagged"] = (
        (df["share_unseen_pairs"] >= X_THRESH) &
        (df[tail_col] >= Y_THRESH) &
        (df["seen_pairs"].fillna(0) >= SEEN_MIN)
    )
    by = df.groupby("pub_year", dropna=True)
    out = pd.DataFrame({
        "pub_year": by.size().index,
        "n_total": by.size().values,
        "n_flagged": by["is_flagged"].sum().values
    })
    out["share_flagged"] = out["n_flagged"] / out["n_total"]
    out.to_csv(f"reports/flagged_by_year_seenmin{SEEN_MIN}.csv", index=False)

    # plots
    plt.figure()
    plt.plot(out["pub_year"], out["n_flagged"])
    plt.title(f'Flagged count per year (seen_pairs ≥ {SEEN_MIN})')
    plt.xlabel("Publication year"); plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(f"plots_year/flagged_count_per_year_seenmin{SEEN_MIN}.png", dpi=150)
    plt.close()

    plt.figure()
    plt.plot(out["pub_year"], out["share_flagged"])
    plt.title(f'Flagged share per year (seen_pairs ≥ {SEEN_MIN})')
    plt.xlabel("Publication year"); plt.ylabel("Share of papers")
    plt.tight_layout()
    plt.savefig(f"plots_year/flagged_share_per_year_seenmin{SEEN_MIN}.png", dpi=150)
    plt.close()

    print(f"[ok] seen_pairs ≥ {SEEN_MIN}: "
          f"saved CSV -> reports/flagged_by_year_seenmin{SEEN_MIN}.csv and plots in plots_year/")


In [ ]:
# Visualize flagged-by-year sensitivity (seen_pairs ≥ 10/20/30)
import os
import pandas as pd
import matplotlib.pyplot as plt

REPORTS = "reports"
OUTDIR  = "plots_year"
os.makedirs(OUTDIR, exist_ok=True)

thresholds = [10, 20, 30]
dfs = {}

# Load available CSVs
for k in thresholds:
    path = os.path.join(REPORTS, f"flagged_by_year_seenmin{k}.csv")
    if os.path.exists(path):
        df = pd.read_csv(path)
        # ensure expected cols exist
        assert {"pub_year","n_total","n_flagged","share_flagged"}.issubset(df.columns), \
            f"Unexpected columns in {path}"
        df = df.sort_values("pub_year")
        dfs[k] = df
    else:
        print(f"[skip] Missing: {path}")

if not dfs:
    raise FileNotFoundError("No flagged_by_year_seenmin*.csv files found in 'reports/'.")

# ---- Plot: Count per year (overlay) ----
plt.figure()
for k, df in dfs.items():
    plt.plot(df["pub_year"], df["n_flagged"], label=f"seen_pairs ≥ {k}")
plt.title('Flagged count per year — "novel + anchored" region')
plt.xlabel("Publication year")
plt.ylabel("Count of flagged papers")
plt.legend()
plt.tight_layout()
p1 = os.path.join(OUTDIR, "flagged_count_per_year_sensitivity.png")
plt.savefig(p1, dpi=150)
plt.show()

# ---- Plot: Share per year (overlay) ----
plt.figure()
for k, df in dfs.items():
    plt.plot(df["pub_year"], df["share_flagged"], label=f"seen_pairs ≥ {k}")
plt.title('Flagged share per year — "novel + anchored" region')
plt.xlabel("Publication year")
plt.ylabel("Share of papers")
plt.legend()
plt.tight_layout()
p2 = os.path.join(OUTDIR, "flagged_share_per_year_sensitivity.png")
plt.savefig(p2, dpi=150)
plt.show()

print("Saved:")
print(" -", p1)
print(" -", p2)

# ---- Quick summary over recent years (last 5 available) ----
for k, df in dfs.items():
    tail = df.tail(5)
    mean_share = tail["share_flagged"].mean()
    total_flag = int(tail["n_flagged"].sum())
    print(f"seen_pairs ≥ {k}: last 5 years — mean share={mean_share:.3f}, total flagged={total_flag}")

In [ ]:
# === Upstream-year sensitivity + cap overlay + venue_type panels ===
# Inputs:
#   features/upstream_uni_year.parquet  (year-based upstream features)
#   clean/nodes_clean.parquet           (to bring in venue_type)
#
# Outputs:
#   plots_year/flagged_share_sensitivity_xy_*.png
#   plots_year/flagged_count_sensitivity_xy_*.png
#   plots_year/flagged_share_vs_cap_overlay.png
#   plots_year/flagged_share_by_venue_type.png
#   reports/flagged_share_sensitivity_xy_*.csv
#   reports/flagged_share_by_venue_type.csv

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FEAT_PATH = "features/upstream_uni_year.parquet"
NODES_PATH = "clean/nodes_clean.parquet"
OUT_PLOTS = "plots_year"
OUT_REPORTS = "reports"
os.makedirs(OUT_PLOTS, exist_ok=True)
os.makedirs(OUT_REPORTS, exist_ok=True)

assert os.path.exists(FEAT_PATH), "Missing features/upstream_uni_year.parquet — run the year-based pipeline first."

# --- Load upstream features ---
u = pd.read_parquet(FEAT_PATH)
u["pub_year"] = pd.to_numeric(u["pub_year"], errors="coerce").astype("Int64")
u["share_unseen_pairs"] = pd.to_numeric(u["share_unseen_pairs"], errors="coerce")
tail_col = "npmi_p10_seen" if "npmi_p10_seen" in u.columns else ("npmi_p10" if "npmi_p10" in u.columns else None)
if tail_col is None:
    raise ValueError("No nPMI tail column found. Expected 'npmi_p10_seen' or 'npmi_p10'.")
u[tail_col] = pd.to_numeric(u[tail_col], errors="coerce")
u["seen_pairs"] = pd.to_numeric(u.get("seen_pairs"), errors="coerce")
u["num_refs"]   = pd.to_numeric(u.get("num_refs"), errors="coerce")
# cap-hit (if not already present)
cap_val = int(u["num_refs"].max()) if u["num_refs"].notna().any() else 120
if "cap_hit" not in u.columns:
    u["cap_hit"] = (u["num_refs"] == cap_val).astype(int)

# --- Bring in venue_type (optional but recommended) ---
if os.path.exists(NODES_PATH):
    nodes = pd.read_parquet(NODES_PATH, columns=["openalex_id","venue_type"])
    u = u.merge(nodes.rename(columns={"openalex_id":"openalex_id"}), on="openalex_id", how="left")
else:
    u["venue_type"] = pd.NA

# Helper: compute per-year summary for a given (x_thresh, y_thresh, seen_min)
def per_year_summary(df, x_thresh, y_thresh, seen_min):
    d = df.dropna(subset=["pub_year","share_unseen_pairs",tail_col]).copy()
    d["is_flagged"] = (
        (d["share_unseen_pairs"] >= x_thresh) &
        (d[tail_col] >= y_thresh) &
        (d["seen_pairs"].fillna(0) >= seen_min)
    )
    g = d.groupby("pub_year", dropna=True)
    out = pd.DataFrame({
        "pub_year": g.size().index.astype(int),
        "n_total": g.size().values,
        "n_flagged": g["is_flagged"].sum().values
    })
    out["share_flagged"] = out["n_flagged"] / out["n_total"]
    return out.sort_values("pub_year")

# ---------- 1) Sensitivity overlays ----------
xy_list = [(0.70, 0.20), (0.75, 0.20), (0.70, 0.25)]
seen_mins = [10, 20, 30]

for (x_t, y_t) in xy_list:
    # gather dfs for all seen_min values
    series = {}
    for smin in seen_mins:
        out = per_year_summary(u, x_t, y_t, smin)
        series[smin] = out
        out.to_csv(os.path.join(OUT_REPORTS, f"flagged_share_sensitivity_xy_{x_t:.2f}_{y_t:.2f}_seen{smin}.csv"), index=False)

    # share overlay
    plt.figure()
    for smin, df in series.items():
        plt.plot(df["pub_year"], df["share_flagged"], label=f"seen_pairs ≥ {smin}")
    plt.title(f'Flagged share per year — sensitivity x≥{x_t}, y≥{y_t}')
    plt.xlabel("Publication year"); plt.ylabel("Share of papers")
    plt.legend()
    plt.tight_layout()
    p_share = os.path.join(OUT_PLOTS, f"flagged_share_sensitivity_xy_{x_t:.2f}_{y_t:.2f}.png")
    plt.savefig(p_share, dpi=150); plt.show()

    # count overlay
    plt.figure()
    for smin, df in series.items():
        plt.plot(df["pub_year"], df["n_flagged"], label=f"seen_pairs ≥ {smin}")
    plt.title(f'Flagged count per year — sensitivity x≥{x_t}, y≥{y_t}')
    plt.xlabel("Publication year"); plt.ylabel("Count of flagged papers")
    plt.legend()
    plt.tight_layout()
    p_count = os.path.join(OUT_PLOTS, f"flagged_count_sensitivity_xy_{x_t:.2f}_{y_t:.2f}.png")
    plt.savefig(p_count, dpi=150); plt.show()

    print("Saved:", p_share, "and", p_count)

# ---------- 2) Cap-hit overlay (baseline spec x=0.70, y=0.20, seen_min=20) ----------
base_x, base_y, base_seen = 0.70, 0.20, 20
flag_y = per_year_summary(u, base_x, base_y, base_seen)
cap_y  = u.dropna(subset=["pub_year"]).groupby("pub_year")["cap_hit"].mean().reset_index()
cap_y["pub_year"] = cap_y["pub_year"].astype(int)

# Align years
merged = flag_y.merge(cap_y, on="pub_year", how="inner").rename(columns={"cap_hit":"share_cap_hit"})

plt.figure()
plt.plot(merged["pub_year"], merged["share_flagged"], label=f'Flagged share (x≥{base_x}, y≥{base_y}, seen≥{base_seen})')
plt.plot(merged["pub_year"], merged["share_cap_hit"], label='Share hitting 120-ref cap')
plt.title('Flagged share vs Cap-hit share by year')
plt.xlabel("Publication year"); plt.ylabel("Share")
plt.legend()
plt.tight_layout()
p_cap = os.path.join(OUT_PLOTS, "flagged_share_vs_cap_overlay.png")
plt.savefig(p_cap, dpi=150); plt.show()
merged.to_csv(os.path.join(OUT_REPORTS, "flagged_share_vs_cap_overlay.csv"), index=False)
print("Saved:", p_cap)

# ---------- 3) Panel by venue_type (share per year) ----------
# Define which venue types to show; keep top 4 by total count + "Other"
vt_counts = u["venue_type"].fillna("NA").value_counts()
top_vt = [v for v in vt_counts.index[:4]]
u["venue_type_panel"] = u["venue_type"].where(u["venue_type"].isin(top_vt), "Other").fillna("Other")

# Flag under baseline thresholds
d = u.dropna(subset=["pub_year","share_unseen_pairs",tail_col]).copy()
d["is_flagged"] = (
    (d["share_unseen_pairs"] >= base_x) &
    (d[tail_col] >= base_y) &
    (d["seen_pairs"].fillna(0) >= base_seen)
)

# Compute share per year within each venue_type_panel
panel = (d.groupby(["venue_type_panel","pub_year"])
           .apply(lambda g: pd.Series({
               "n_total": len(g),
               "n_flagged": int(g["is_flagged"].sum()),
               "share_flagged": float(g["is_flagged"].mean())
           }))
           .reset_index()
           .sort_values(["venue_type_panel","pub_year"]))

panel.to_csv(os.path.join(OUT_REPORTS, "flagged_share_by_venue_type.csv"), index=False)

# Plot panels
cats = panel["venue_type_panel"].unique().tolist()
n = len(cats)
cols = min(3, n)
rows = (n + cols - 1) // cols

plt.figure(figsize=(5*cols, 3.8*rows))
for i, vt in enumerate(cats, 1):
    ax = plt.subplot(rows, cols, i)
    sub = panel[panel["venue_type_panel"] == vt]
    ax.plot(sub["pub_year"], sub["share_flagged"])
    ax.set_title(vt)
    ax.set_xlabel("Year"); ax.set_ylabel("Share flagged")
    ax.set_ylim(0, max(0.3, sub["share_flagged"].max()*1.1))
plt.suptitle(f'Flagged share by venue_type (x≥{base_x}, y≥{base_y}, seen≥{base_seen})')
plt.tight_layout(rect=[0,0,1,0.95])
p_panel = os.path.join(OUT_PLOTS, "flagged_share_by_venue_type.png")
plt.savefig(p_panel, dpi=150); plt.show()
print("Saved:", p_panel)

In [ ]:
# Count & top-10 for "novel + anchored" papers
# Region: share_unseen_pairs ≥ X_THRESH  AND  npmi_p10_seen ≥ Y_THRESH
# Guardrail: seen_pairs ≥ SEEN_MIN

import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

FEAT_PATH  = "features/upstream_uni_year.parquet"   # year-based upstream features
NODES_PATH = "clean/nodes_clean.parquet"            # to enrich with titles/venues (optional but recommended)
OUT_DIR    = "reports"
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.exists(FEAT_PATH), "Missing features/upstream_uni_year.parquet. Run the year-based upstream pipeline."

# Thresholds (tune here)
X_THRESH = 0.70   # min share of NEW pairs
Y_THRESH = 0.20   # min nPMI tail (seen-only)
SEEN_MIN = 20     # coverage floor

# Load upstream features
u = pd.read_parquet(FEAT_PATH)

# Choose the tail column (prefer seen-only)
tail_col = "npmi_p10_seen" if "npmi_p10_seen" in u.columns else ("npmi_p10" if "npmi_p10" in u.columns else None)
if tail_col is None:
    raise ValueError("Expected 'npmi_p10_seen' (or 'npmi_p10') in upstream features.")

# Coerce needed columns
u["pub_year"] = pd.to_numeric(u.get("pub_year"), errors="coerce").astype("Int64")
u["share_unseen_pairs"] = pd.to_numeric(u["share_unseen_pairs"], errors="coerce")
u[tail_col] = pd.to_numeric(u[tail_col], errors="coerce")
u["seen_pairs"] = pd.to_numeric(u.get("seen_pairs"), errors="coerce").fillna(0)

# Filter to region + coverage floor
flag = (
    (u["share_unseen_pairs"] >= X_THRESH) &
    (u[tail_col] >= Y_THRESH) &
    (u["seen_pairs"] >= SEEN_MIN)
)
flagged = u.loc[flag].copy()
total = len(u)
n_flag = len(flagged)
frac = n_flag / total if total else np.nan

print(f"Total papers evaluated: {total:,}")
print(f'Flagged in "novel + anchored" region: {n_flag:,} ({frac:.2%}) '
      f'[x≥{X_THRESH}, y≥{Y_THRESH}, seen_pairs≥{SEEN_MIN}]')

# Optional: enrich with titles / venues / doi
if os.path.exists(NODES_PATH):
    meta_cols = ["openalex_id","title","venue_name","venue_type","doi","pub_year"]
    # Use pyarrow to get column names without loading data
    nodes_columns = pq.ParquetFile(NODES_PATH).schema.names
    nodes = pd.read_parquet(NODES_PATH, columns=[c for c in meta_cols if c in nodes_columns])
    flagged = flagged.merge(nodes.rename(columns={"openalex_id":"openalex_id"}), on="openalex_id", how="left")

# Rank by a composite upstream score (anchored novelty first)
# z-score helpers
def z(x):
    x = pd.to_numeric(x, errors="coerce")
    mu, sd = x.mean(), x.std()
    return (x - mu) / sd if sd and not np.isnan(sd) else pd.Series(np.zeros(len(x)), index=x.index)

flagged["z_npmi_tail"] = z(flagged[tail_col])
flagged["z_share_new"] = z(flagged["share_unseen_pairs"])
flagged["upstream_score"] = 0.7 * flagged["z_npmi_tail"] + 0.3 * flagged["z_share_new"]

# Order and select columns for display
display_cols = [
    "openalex_id", "title", "pub_year", "venue_name", "venue_type", "doi",
    "share_unseen_pairs", tail_col, "seen_pairs", "num_refs", "upstream_score"
]
display_cols = [c for c in display_cols if c in flagged.columns]

top10 = (flagged
         .sort_values("upstream_score", ascending=False)
         .loc[:, display_cols]
         .head(10))

print("\nTop 10 flagged papers (by upstream_score):")
pd.set_option("display.max_colwidth", 120)
print(top10.to_string(index=False))

# Save full flagged list and top 10
flagged_out = os.path.join(OUT_DIR, f"flagged_novel_anchored_all_x{X_THRESH}_y{Y_THRESH}_seen{SEEN_MIN}.csv")
top10_out   = os.path.join(OUT_DIR, f"flagged_novel_anchored_top10_x{X_THRESH}_y{Y_THRESH}_seen{SEEN_MIN}.csv")
flagged.loc[:, display_cols].to_csv(flagged_out, index=False)
top10.to_csv(top10_out, index=False)
print(f"\nSaved full flagged list -> {flagged_out}")
print(f"Saved top 10 -> {top10_out}")